In [ ]:
# Compare: time the viewer before and after a change, in the same Colab session.
# Colab's speed drifts from one session to the next, so the old and the new viewer are
# built here side by side and play the same walk in turns. Needs a T4 GPU runtime.
import csv, os, shutil, statistics, subprocess, sys

REPO_URL = "https://github.com/ShreyasxAnand/pointwalk"
BRANCH = "dev"
BEFORE = "6929088"   # the commit to compare against: the viewer before the change
ROUNDS = 2           # each version plays the walk this many times, taking turns
FLAGS = ["-O2", "-arch=sm_75", "-fmad=false"]  # -fmad=false: CPU and GPU match bit for bit

repo = "/content/" + REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
build = os.path.join(repo, "build_compare")


def sh(*cmd, cwd=None, quiet=False):
    """Run a command, print its output, stop the cell if it fails."""
    print("$ " + " ".join(cmd), flush=True)
    r = subprocess.run(cmd, cwd=cwd, text=True,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if not quiet or r.returncode != 0:
        print(r.stdout, flush=True)
    if r.returncode != 0:
        raise SystemExit(f"FAILED (exit {r.returncode}): {' '.join(cmd)}")
    return r.stdout


# 1. Get the code.
if not os.path.isdir(os.path.join(repo, ".git")):
    sh("git", "clone", "--branch", BRANCH, REPO_URL, repo)
else:
    sh("git", "fetch", "origin", BRANCH, cwd=repo)
    sh("git", "checkout", BRANCH, cwd=repo)
    sh("git", "pull", "--ff-only", "origin", BRANCH, cwd=repo)
after = sh("git", "log", "-1", "--format=%h", cwd=repo, quiet=True).strip()
sh("git", "log", "-1", "--format=%h %ad %s", "--date=iso", cwd=repo)   # the "after" version
sh("nvidia-smi", "--query-gpu=name", "--format=csv,noheader")

# 2. Put a copy of the old source next to the new one.
shutil.rmtree(build, ignore_errors=True)
os.makedirs(build)
old_source = os.path.join(build, "before_source")
sh("git", "worktree", "prune", cwd=repo)
sh("git", "worktree", "add", "--detach", old_source, BEFORE, cwd=repo, quiet=True)

# 3. Compile. The new project program runs once: it checks GPU against CPU and saves the scene.
folders = {"before": os.path.join(build, "before"), "after": os.path.join(build, "after")}
for folder in folders.values():
    os.makedirs(folder)
sh("nvcc", *FLAGS, os.path.join(old_source, "src", "viewer.cu"), "-o", os.path.join(folders["before"], "viewer"))
sh("nvcc", *FLAGS, os.path.join(repo, "src", "viewer.cu"), "-o", os.path.join(folders["after"], "viewer"))
sh("nvcc", *FLAGS, os.path.join(repo, "src", "project.cu"), "-o", os.path.join(folders["after"], "project"))
out = sh("./project", cwd=folders["after"], quiet=True)
print("\n".join(line for line in out.splitlines() if "differ" in line), "\n")
scene = os.path.join(folders["after"], "scene.bin")

# 4. Both viewers play the same walk in turns. Keep the middle time of each run.
COLUMNS = [("kernel_ms", "the two kernels"),
           ("gpu_picture_ms", "whole picture on GPU"),
           ("round_trip_ms", "handed to script")]
results = {"before": [], "after": []}
for round_number in range(1, ROUNDS + 1):
    for version in ["before", "after"]:
        out_dir = os.path.join(folders[version], f"round{round_number}")
        os.makedirs(out_dir)
        sh(sys.executable, os.path.join(repo, "viewer", "replay.py"),
           "--renderer", os.path.join(folders[version], "viewer"),
           "--scene", scene, "--out", out_dir, quiet=True)
        with open(os.path.join(out_dir, "walk_timing.csv")) as f:
            rows = list(csv.DictReader(f))
        results[version].append([statistics.median(float(r[key]) for r in rows) for key, _ in COLUMNS])

# 5. The table: one line per run, then how much the new version changed each time.
names = {"before": f"before {BEFORE}", "after": f"after  {after}"}
print()
print(" " * 26 + "".join(f"{title:>24}" for _, title in COLUMNS))
for round_number in range(ROUNDS):
    for version in ["before", "after"]:
        times = results[version][round_number]
        print(f"{names[version]:<17} run {round_number + 1}   " + "".join(f"{t:>21.3f} ms" for t in times))
change = []
for column in range(len(COLUMNS)):
    old = statistics.mean(run[column] for run in results["before"])
    new = statistics.mean(run[column] for run in results["after"])
    change.append((new - old) / old * 100.0)
print(f"{'change (minus = faster)':<26}" + "".join(f"{c:>+22.1f} %" for c in change))
